# BIY7121 — Evaluación 3 — CortesDiego

## Insight seleccionado

**Proyecto grupal:** Dog Aging Project (DAP) 2024 — segmentación de perros según comportamiento, salud y trato del tutor.

**Insight (sección 8 del notebook grupal):**

> El nivel de cuidado preventivo y la actividad física que provee el tutor se asocian con la carga de enfermedades del perro. Los perfiles de menor actividad y menor cuidado preventivo concentran mayor proporción de salud/enfermedad alta.

**Modelo desplegado:** clasificación binaria `multimorbidity_flag` (perro con ≥2 condiciones de salud) a partir de hábitos del tutor y controles básicos del perro.

**Métrica principal:** F1-score (validación cruzada 5-fold).

## 1. Carga de datos

Coloque `dap_consolidado.csv` en `data/dap_consolidado.csv` (descargado desde el notebook grupal en Google Drive).

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

ROOT = Path('.').resolve()
if (ROOT / 'BIY7121_EV3_CortesDiego').exists():
    ROOT = ROOT / 'BIY7121_EV3_CortesDiego'
elif ROOT.name != 'BIY7121_EV3_CortesDiego':
    candidate = Path.cwd() / 'BIY7121_EV3_CortesDiego'
    if candidate.exists():
        ROOT = candidate

sys.path.insert(0, str(ROOT / 'modelo_y_preprocesamiento'))
from preprocess import prepare_dataframe_features, FEATURE_ORDER, PARAM_GRID, RANDOM_STATE
from train_and_export import load_data, generate_synthetic_dap

DATA_PATH = ROOT / 'data' / 'dap_consolidado.csv'
if DATA_PATH.exists():
    df = pd.read_csv(DATA_PATH, low_memory=False)
    print('Datos reales:', df.shape)
else:
    df = load_data()
    print('Datos cargados (sintéticos si no hay CSV):', df.shape)

df.head(3)

## 2. Preprocesamiento y variables

| Variable API | Origen DAP |
|---|---|
| `multimorbidity_flag` (target) | `salud_num_condiciones >= 2` |
| `preventive_care_score` | z-score promedio de cepillado, examen dental, grooming, dieta orgánica |
| `pa_activity_level` | `pa_activity_level` |
| `pa_avg_activity_intensity` | `pa_avg_activity_intensity` |
| `total_daily_active_minutes` | `pa_avg_daily_active_minutes` |
| `dental_brushing_freq` | `mp_dental_brushing_frequency` |
| `daily_supplements` | `mp_dental_examination_frequency` (proxy) |
| `diet_primary` | `df_primary_diet_component_organic` |
| `weight_kg` | `dd_weight_lbs × 0.453592` |
| `age_derived` | `dd_edad_anios` |

In [ ]:
X, y, medians, preventive_stats = prepare_dataframe_features(df)
print('Features:', FEATURE_ORDER)
print('Tasa multimorbilidad:', round(y.mean(), 3))
print(y.value_counts())
X.describe().round(2)

## 3. EDA breve

In [ ]:
eda = X.copy()
eda['multimorbidity_flag'] = y.values
corr = eda.corr()['multimorbidity_flag'].drop('multimorbidity_flag').sort_values(ascending=False)
print('Correlación con multimorbilidad:')
print(corr.round(3))

fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(x=corr.values, y=corr.index, ax=ax, palette='RdYlGn_r')
ax.set_title('Correlación de predictores con multimorbilidad')
ax.set_xlabel('Correlación de Pearson')
plt.tight_layout()
plt.show()

## 4. Comparación de modelos, validación cruzada y GridSearchCV

In [ ]:
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay, f1_score, roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, cross_val_score, train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

from preprocess import save_metadata, artifact_dir

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)

modelos = {
    'Naive Bayes Gaussian': make_pipeline(StandardScaler(), GaussianNB()),
    'Árbol de Decisión': DecisionTreeClassifier(
        max_depth=4, min_samples_leaf=50, class_weight='balanced', random_state=RANDOM_STATE
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=150, max_depth=10, min_samples_leaf=10,
        class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1
    ),
}

resultados = []
for nombre, modelo in modelos.items():
    cv = cross_val_score(modelo, X_train, y_train, cv=5, scoring='f1', n_jobs=-1)
    modelo.fit(X_train, y_train)
    pred = modelo.predict(X_test)
    proba = modelo.predict_proba(X_test)[:, 1] if hasattr(modelo, 'predict_proba') else None
    resultados.append({
        'Modelo': nombre,
        'F1_cv_mean': cv.mean(),
        'F1_cv_std': cv.std(),
        'F1_test': f1_score(y_test, pred),
        'Accuracy_test': accuracy_score(y_test, pred),
        'ROC_AUC_test': roc_auc_score(y_test, proba) if proba is not None else np.nan,
    })

tabla = pd.DataFrame(resultados).sort_values('F1_cv_mean', ascending=False)
display(tabla.round(4))

### Selección justificada del mejor modelo

Se elige **Random Forest** porque obtiene el mejor o comparable F1 en validación cruzada, supera al azar y entrega importancia de variables interpretable, coherente con la sección 8.4 del proyecto grupal.

In [ ]:
rf = RandomForestClassifier(class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1)
grid = GridSearchCV(rf, PARAM_GRID, cv=5, scoring='f1', n_jobs=-1, refit=True)
grid.fit(X_train, y_train)

print('Mejores hiperparámetros:', grid.best_params_)
print('Mejor F1 CV:', round(grid.best_score_, 4))

best_model = grid.best_estimator_
best_model.fit(X, y)

y_pred = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred, target_names=['Sin multimorbilidad', 'Multimorbilidad']))

fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred),
    display_labels=['Sin multimorb.', 'Multimorb.']).plot(ax=ax, cmap='Blues')
plt.title('Matriz de confusión — Random Forest')
plt.tight_layout()
plt.show()

imp = pd.DataFrame({'variable': FEATURE_ORDER, 'importancia': best_model.feature_importances_})
imp = imp.sort_values('importancia', ascending=False)
display(imp.round(4))

## 5. Exportación del modelo y preprocesamiento

In [ ]:
out = artifact_dir(ROOT / 'modelo_y_preprocesamiento')
joblib.dump(best_model, out / 'model.joblib')

metric_ref = f'F1_cv={grid.best_score_:.4f}'
save_metadata(
    medians,
    preventive_stats,
    {
        'best_params': grid.best_params_,
        'best_cv_f1': float(grid.best_score_),
        'f1_test': float(f1_score(y_test, y_pred)),
        'roc_auc_test': float(roc_auc_score(y_test, y_proba)),
    },
    metric_ref,
    out,
)

print('Exportado en:', out)
print('Archivos:', [p.name for p in out.iterdir()])